In [ ]:
from workshop_helpers import run_environment_check

# performance_mode=False keeps Genesis in ndarray mode: kernels are not specialised on the
# scene shape, so train/eval/video scenes share one compiled set (slower physics, faster startup).
run_environment_check(performance_mode=True)

# ROSCon Workshop: From Flat Ground to Stairs — Training

A stable Unitree G1 flat-ground policy is the starting point. This notebook configures a gentle
stair curriculum and fine-tunes that policy with PPO.

Hosted by **AMD and Robotec.ai**, the workshop runs on an **AMD Strix Halo mini-PC**. The notebook
contains the configuration and training steps and uses the `gslab` library to connect Genesis
simulation, terrain generation, and reinforcement learning in one reproducible workflow.

## Learning outcomes

By the end of this notebook you will be able to:

* describe the high-level ingredients used to train the flat-ground baseline;
* identify which environment, reward, and curriculum settings change for stairs;
* explain why the flat policy can be fine-tuned instead of trained again from scratch; and
* launch a repeatable fine-tuning run that writes to a fixed checkpoint path.

## Setup

The infrastructure setup is documented in [`../INSTALL.md`](../INSTALL.md). Select the
**gslab ROSCon (ROCm 7.13)** kernel and run cells in order.

The first cell is an executable infrastructure check. It must print `PASS` before training starts.
The training cell remains busy while PPO runs, but the explanatory sections remain readable.

In [ ]:
import gslab.tasks  # registers every task  # noqa: F401
from gslab.tasks.registry import load_env_cfg, load_rl_cfg
from workshop_helpers import bootstrap_workshop, print_device_summary

WORKSHOP_ROOT = bootstrap_workshop()
DEVICE = "cuda"  # PyTorch's name for the GPU device; it is the AMD GPU on a ROCm build

print_device_summary(WORKSHOP_ROOT)


In [ ]:
from workshop_helpers import require_checkpoint
from workshop_training import run_training


## Configure the fine-tuning run

The paths are intentionally stable—there is no timestamp to copy between notebooks. Re-running
this workshop overwrites `logs/roscon_stairs/model_stairs.pt`.

The code loads only the actor from the flat checkpoint. The critic and optimizer start fresh,
which lets value estimation adapt quickly to the new terrain distribution while preserving the
useful walking gait.

In [ ]:
TASK = "Unitree-G1-Stairs-Easy"
FLAT_TASK = "Unitree-G1-Flat"
BASELINE = require_checkpoint(WORKSHOP_ROOT / "checkpoints/g1_flat_baseline.pt")
RUN_DIR = WORKSHOP_ROOT / "logs/roscon_stairs"
OUTPUT_CHECKPOINT = RUN_DIR / "model_stairs.pt"

NUM_ENVS = 1024  # one Genesis step costs ~0.25 s with 1,024 robots but ~1 s with 2,048 on a Strix Halo
STEPS_PER_UPDATE = 48  # rollout length per robot; 1,024 x 48 gives each PPO update the same 49k samples as the
#                        2,048 x 24 recipe the reference checkpoint was trained with, at half the step cost
ITERATIONS = 200  # upper bound on PPO updates; the time budget below is what actually ends the run (~115 updates)
TIME_BUDGET_MIN = 25  # hard wall-clock limit for the training cell, scene build included

flat_cfg = load_env_cfg(FLAT_TASK)
env_cfg = load_env_cfg(TASK)
env_cfg.scene.num_envs = NUM_ENVS
# Flat margin around the grid. Episodes are 20 s at up to 1 m/s, so a robot that
# clears its stairs keeps walking on flat ground instead of off the heightfield.
env_cfg.scene.terrain.terrain_generator.border_width = 20.0
env_cfg.seed = 0

agent_cfg = load_rl_cfg(TASK)
agent_cfg.max_iterations = ITERATIONS
agent_cfg.num_steps_per_env = STEPS_PER_UPDATE
agent_cfg.save_interval = 50
agent_cfg.logger = "tensorboard"
agent_cfg.algorithm.learning_rate = 5.0e-4

flat_actor_terms = tuple(flat_cfg.observations["actor"].terms)
stairs_actor_terms = tuple(env_cfg.observations["actor"].terms)
assert flat_actor_terms == stairs_actor_terms
assert tuple(flat_cfg.actions) == tuple(env_cfg.actions)

terrain_cfg = env_cfg.scene.terrain.terrain_generator
reward_changes = {
    name: (flat_cfg.rewards[name].weight, term.weight)
    for name, term in env_cfg.rewards.items()
    if name in flat_cfg.rewards and flat_cfg.rewards[name].weight != term.weight
}

RUN_DIR.mkdir(parents=True, exist_ok=True)
print(f"baseline          : {BASELINE.relative_to(WORKSHOP_ROOT)}")
print(f"output checkpoint : {OUTPUT_CHECKPOINT.relative_to(WORKSHOP_ROOT)}")
print(f"parallel robots   : {NUM_ENVS:,}")
print(f"PPO updates       : {ITERATIONS}")
print(f"actor interface   : unchanged ({len(flat_actor_terms)} observation terms)")
print(f"terrain types     : {list(terrain_cfg.sub_terrains)}")
print(f"difficulty levels : {terrain_cfg.num_rows}")
print(f"curriculum        : {list(env_cfg.curriculum)}")
print(f"reward weights    : {reward_changes}")


## Start training

This is the long-running cell. Genesis first builds the scene quietly (about 1.5 minutes), then PPO
updates the policy from 1,024 parallel robots, 48 control steps each per update. One update takes
about 12 seconds on a Strix Halo, so the **25-minute budget** below allows roughly 115 updates;
the loop checks after every update whether one more fits and otherwise stops and saves the
checkpoint. A live progress bar and reward curves replace the verbose text log; full metrics are
still written to TensorBoard under `logs/roscon_stairs/`. The final actor is written to the fixed
path printed above.

What to expect from a 25-minute run: falls across the whole terrain grid drop from about 24% for
the flat baseline to about 1%, with a slower, more careful gait (about 0.45 m/s against the
0.8 m/s command). The hardest box staircase (level 5) is usually still not solved in this time. `checkpoints/g1_stairs_easy_trained.pt` is exactly such a run; the reference
`checkpoints/g1_ref.pt`, trained for over an hour with 2,048 robots, shows what more training buys.


In [ ]:
run_training(
    env_cfg=env_cfg,
    task=TASK,
    baseline=BASELINE,
    run_dir=RUN_DIR,
    agent_cfg=agent_cfg,
    output_checkpoint=OUTPUT_CHECKPOINT,
    iterations=ITERATIONS,
    workshop_root=WORKSHOP_ROOT,
    time_budget_s=TIME_BUDGET_MIN * 60,
)


## What flat walking requires

The baseline already walks. On a plane the policy has learned to:

* keep balance while one foot is in the air;
* hold a steady step rhythm;
* track commanded forward, sideways and turning speed;
* stay upright, lift and place feet cleanly, act smoothly, respect joint limits;
* do all of it from body sense alone: no camera, no height scan;
* cope with random ground friction.

Stairs keep all of this and add the problem that the ground is no longer where the feet expect it.


## Upgrade 1: same robot, different world

* Same senses, same actuators, same 50 Hz decisions: the flat weights load unchanged.
* The plane becomes a generated landscape.
* Physics runs at half the time step so hard, vertical step contacts stay stable.


## Upgrade 2: a difficulty ladder

* A grid of four ground types (flat, pyramid stairs, rough, real block stairs) times six
  difficulty rows.
* Robots start on the easiest rows; cross the patch and you are promoted, cover less than half the
  commanded distance and you are demoted.
* Flat ground stays in the mix so the gait is not forgotten.
* Block stairs have true vertical risers; the staircase climbs, then descends, so nobody walks off
  an edge.


## Upgrade 3: adapt the learning signal

* Foot clearance is measured from the ground under the foot and the target is raised to step
  height; only swing feet count, so standing still cannot game it.
* Torso tilt is penalised half as much: leaning into a climb is allowed.
* Difficulty itself is not rewarded. It only changes what the robots experience.


## Upgrade 4: fine-tune carefully

* Half the usual learning rate, so early falls do not wipe out the gait.
* Only the walking behaviour is inherited; the value estimate starts fresh.
* Result: far fewer falls than the baseline, at a slower walk. More training buys the speed back.


## Key takeaways

* Transfer is possible because the flat and stairs tasks share an actor interface.
* Terrain curriculum controls the experience presented to PPO; rewards control which behavior is
  reinforced within that experience.
* The smaller timestep stabilizes rigid stair contacts without changing the 50 Hz policy rate.
* A reduced learning rate protects the useful baseline gait during adaptation.
